##### 01 - Data Understanding


In [1]:
from pathlib import Path
import pandas as pd

RAW = Path('../data/raw')
TXN_FILE = RAW / 'card_transaction.v1.csv'
USERS_FILE = RAW / 'sd254_users.csv'
CARDS_FILE = RAW / 'sd254_cards.csv'

In [2]:
users = pd.read_csv(USERS_FILE)
cards = pd.read_csv(CARDS_FILE)

print('Users: ', users.shape)
print('Cards: ', cards.shape)
print(users.dtypes)
print(cards.dtypes)

Users:  (2000, 18)
Cards:  (6146, 13)
Person                             str
Current Age                      int64
Retirement Age                   int64
Birth Year                       int64
Birth Month                      int64
Gender                             str
Address                            str
Apartment                      float64
City                               str
State                              str
Zipcode                          int64
Latitude                       float64
Longitude                      float64
Per Capita Income - Zipcode        str
Yearly Income - Person             str
Total Debt                         str
FICO Score                       int64
Num Credit Cards                 int64
dtype: object
User                     int64
CARD INDEX               int64
Card Brand                 str
Card Type                  str
Card Number              int64
Expires                    str
CVV                      int64
Has Chip                   st

In [3]:
columns = pd.read_csv(TXN_FILE, nrows=0).columns
total_rows = 0
blank_counts = pd.Series(0, index=columns)

for chunk in pd.read_csv(TXN_FILE, dtype=str, keep_default_na=False, 
                         chunksize=200_000):
    total_rows += len(chunk)
    blank_counts += (chunk == '').sum()

print(f'Total rows: {total_rows:,}')
print(blank_counts)

Total rows: 24,386,900
User                     0
Card                     0
Year                     0
Month                    0
Day                      0
Time                     0
Amount                   0
Use Chip                 0
Merchant Name            0
Merchant City            0
Merchant State     2720821
Zip                2878135
MCC                      0
Errors?           23998469
Is Fraud?                0
dtype: int64


In [4]:
from collections import Counter

use_chip = Counter()
is_fraud = Counter()
errors = Counter()
years = Counter()
state_lengths = Counter()
online_rows = 0
negative_amounts = 0
mcc_codes = set()

for chunk in pd.read_csv(TXN_FILE, keep_default_na=False, chunksize=200_000):
    use_chip.update(chunk['Use Chip'])
    is_fraud.update(chunk['Is Fraud?'])
    errors.update(chunk['Errors?'])
    years.update(chunk['Year'])
    state_lengths.update(chunk['Merchant State'].str.len())
    online_rows += (chunk['Merchant City'].str.strip() == 'ONLINE').sum()
    negative_amounts += (chunk['Amount'].str.contains('-').sum())
    mcc_codes.update(chunk['MCC'].unique())

print("Use Chip:", dict(use_chip))
print("Is Fraud?:", dict(is_fraud))
print("Years:", min(years), "to", max(years), "| distinct:", len(years))
print("Online rows (city text):", online_rows)
print("Negative amounts:", negative_amounts)
print("Distinct MCC:", len(mcc_codes))
print("Merchant State length counts:", dict(state_lengths))
print("Distinct Errors? values:", len(errors))
for value, count in errors.most_common(15):
    print(repr(value), count)

Use Chip: {'Swipe Transaction': 15386082, 'Online Transaction': 2713220, 'Chip Transaction': 6287598}
Is Fraud?: {'No': 24357143, 'Yes': 29757}
Years: 1991 to 2020 | distinct: 30
Online rows (city text): 2720821
Negative amounts: 1244689
Distinct MCC: 109
Merchant State length counts: {2: 21508765, 0: 2720821, 11: 10053, 7: 14527, 5: 26681, 6: 81051, 18: 2754, 8: 5060, 12: 1041, 9: 3691, 14: 8796, 4: 1009, 10: 1782, 16: 42, 20: 552, 13: 87, 15: 31, 21: 16, 32: 12, 22: 60, 17: 3, 19: 29, 24: 37}
Distinct Errors? values: 24
'' 23998469
'Insufficient Balance,' 242783
'Bad PIN,' 58918
'Technical Glitch,' 48157
'Bad Card Number,' 13321
'Bad CVV,' 10740
'Bad Expiration,' 10716
'Bad Zipcode,' 2079
'Bad PIN,Insufficient Balance,' 581
'Insufficient Balance,Technical Glitch,' 457
'Bad PIN,Technical Glitch,' 128
'Bad Card Number,Insufficient Balance,' 122
'Bad CVV,Insufficient Balance,' 89
'Bad Expiration,Insufficient Balance,' 78
'Bad Card Number,Bad CVV,' 60
